In [11]:
import pandas as pd
df = pd.read_csv('서울교통공사_역별 일별 시간대별 승하차인원_20211231.csv', encoding='cp949')
df = df[df['구분'] == '승차']

# 1호선부터 8호선까지만 필터링 (9호선이나 국철 제거)
df = df[df['호선'].isin([1, 2, 3, 4, 5, 6, 7, 8])]

df_result = df.groupby('날짜')['합 계'].sum().reset_index()
df_result.columns = ['날짜', '일별전체승차량']


df_result.to_csv('2021.csv', index=False, encoding='utf-8-sig')

print(" 전처리 완료! '최종_하루별_총승차량.csv'")
display(df_result.head()) # 상위 5일치 결과 출력해서 확인하기

 전처리 완료! '최종_하루별_총승차량.csv'


,날짜,일별전체승차량
0,2021-01-01,968534
1,2021-01-02,1535446
2,2021-01-03,1168769
3,2021-01-04,3455939
4,2021-01-05,3464498


In [12]:
import pandas as pd

# 1. 2022년 데이터 불러오기
df = pd.read_csv('2.서울교통공사_역별 일별 시간대별 승하차인원 정보_20221231.csv', encoding='cp949')

# 2. '승차' 데이터만 남기기
df = df[df['승하차구분'] == '승차'].copy()



time_cols = [col for col in df.columns if '시' in col]
df['합계'] = df[time_cols].sum(axis=1)


df_result = df.groupby('수송일자')['합계'].sum().reset_index()


df_result.columns = ['날짜', '일별전체승차량']

print("✅ 없는 합계도 파이썬으로 뚝딱 창조했습니다!")
display(df_result.head())
df_result.to_csv('2022.csv', index=False, encoding='utf-8-sig')

✅ 없는 합계도 파이썬으로 뚝딱 창조했습니다!


/tmp/ipykernel_450/3756153587.py:4: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('2.서울교통공사_역별 일별 시간대별 승하차인원 정보_20221231.csv', encoding='cp949')


,날짜,일별전체승차량
0,2022-01-01,1591251.0
1,2022-01-02,1808051.0
2,2022-01-03,3958648.0
3,2022-01-04,4106543.0
4,2022-01-05,4124783.0


In [13]:
import pandas as pd

# 1. 파일 불러오기
df = pd.read_csv('서울교통공사_1~8호선일별역별시간대별승하차인원_2019년.csv', encoding='cp949')

# 2. '승차' 데이터만 남기기
df = df[df['구분'] == '승차'].copy()



# '06시 이전'이 시작되는 6번째 컬럼(인덱스 5)부터 끝까지 싹 다 지정해서 더하기
time_cols = df.columns[5:]
df['합계'] = df[time_cols].sum(axis=1)

#'날짜' 기준으로 묶기
df_result = df.groupby('날짜')['합계'].sum().reset_index()

#
df_result.columns = ['날짜', '일별전체승차량']

print("압축 완료!")
display(df_result.head())

df_result.to_csv('2019.csv', index=False, encoding='utf-8-sig')

압축 완료!


,날짜,일별전체승차량
0,2019-01-01,2169874
1,2019-01-02,5067609
2,2019-01-03,5255830
3,2019-01-04,5485567
4,2019-01-05,3921397


In [14]:
import pandas as pd

# 1. (2021, 2022년 전체 승차량만 불러오기)
total_21 = pd.read_csv('2021.csv', encoding='utf-8')
total_22 = pd.read_csv('2022.csv', encoding='utf-8')

total_21.columns = ['날짜', '일별전체승차량']
total_22.columns = ['날짜', '일별전체승차량']

df_total = pd.concat([total_21, total_22], ignore_index=True)


# 2. 종합 무임승차 파일 불러오기
df_free = pd.read_excel('[19-25]일별우대권승차.xlsx')
df_free.columns = ['날짜', '일별무임승차량']


df_free['날짜'] = pd.to_datetime(df_free['날짜']).dt.strftime('%Y-%m-%d')


# 3. OUTER JOIN (합집합 병합)
df_merged = pd.merge(df_free, df_total, on='날짜', how='outer')


# 4. 연도별 고유 비율 계산 및 결측치 채우기
df_merged['날짜'] = pd.to_datetime(df_merged['날짜'])
df_merged['연도'] = df_merged['날짜'].dt.year

valid_data = df_merged.dropna(subset=['일별무임승차량', '일별전체승차량'])
yearly_ratio = valid_data.groupby('연도')['일별무임승차량'].sum() / valid_data.groupby('연도')['일별전체승차량'].sum()

print("\n 21년, 22년 결측치를 채울 고유 무임승차 비율:")
print(yearly_ratio)

df_merged['맞춤비율'] = df_merged['연도'].map(yearly_ratio)

# 비어있는(NaN) 칸 채우기
df_merged['일별무임승차량'] = df_merged['일별무임승차량'].fillna(df_merged['일별전체승차량'] * df_merged['맞춤비율'])


# 5. 마무리 정리 및 마스터 파일 저장
df_merged['일별무임승차량'] = df_merged['일별무임승차량'].astype(int)
df_merged = df_merged.sort_values('날짜').reset_index(drop=True)
df_merged['날짜'] = df_merged['날짜'].dt.strftime('%Y-%m-%d')

# 날짜와 무임승차량 딱 두 컬럼만 추출
df_final = df_merged[['날짜', '일별무임승차량']]

df_final.to_csv('최종_완벽본_일별무임승차량(19-25).csv', index=False, encoding='utf-8-sig')

print("\n 날짜 포맷 에러 해결 및 병합 완료!")
display(df_final.head(10))


 21년, 22년 결측치를 채울 고유 무임승차 비율:
연도
2021    0.148495
2022    0.158455
dtype: float64

 날짜 포맷 에러 해결 및 병합 완료!


,날짜,일별무임승차량
0,2019-01-01,324283
1,2019-01-02,722134
2,2019-01-03,765807
3,2019-01-04,784641
4,2019-01-05,633946
5,2019-01-06,475875
6,2019-01-07,791914
7,2019-01-08,795932
8,2019-01-09,755806
9,2019-01-10,798343


In [15]:
import pandas as pd
import glob
file_list = glob.glob("기간별_일평균_대기환경_정보_*.csv")

df_list = []
for file in file_list:

    try:
        temp_df = pd.read_csv(file, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            temp_df = pd.read_csv(file, encoding='cp949')
        except UnicodeDecodeError:
            temp_df = pd.read_csv(file, encoding='euc-kr')


    for col in temp_df.columns:
        if '초미세' in col:
            temp_df.rename(columns={col: '초미세먼지'}, inplace=True)
        elif '미세먼지' in col:  # '초미세'가 먼저 걸러지므로 안전합니다.
            temp_df.rename(columns={col: '미세먼지'}, inplace=True)
        elif '측정일시' in col or '날짜' in col:
            temp_df.rename(columns={col: '측정일시'}, inplace=True)

    df_list.append(temp_df)

# 이제 모든 파일의 컬럼명이 ['측정일시', '미세먼지', '초미세먼지']로 똑같아졌으므로 이쁘게 합쳐집니다
df_all_dust = pd.concat(df_list, ignore_index=True)
print(f" -> 총 {len(file_list)}개의 파일 컬럼명 자동 통일 및 합체 완료!")


# =========================================================
# [2단계] 서울시 일평균으로 압축
# =========================================================
# 이미 컬럼 이름을 위에서 '측정일시', '미세먼지', '초미세먼지'로 고정해 뒀기 때문에 에러가 안 납니다.
df_dust_sub = df_all_dust[['측정일시', '미세먼지', '초미세먼지']]

# 25개 구 데이터를 하루 평균으로 압축
df_dust_daily = df_dust_sub.groupby('측정일시').mean().reset_index()
df_dust_daily = df_dust_daily.round(1)


# =========================================================
# [3단계] 날짜 모양 세탁 및 최종 정돈
# =========================================================
df_dust_daily['측정일시'] = pd.to_datetime(df_dust_daily['측정일시'].astype(str)).dt.strftime('%Y-%m-%d')
df_dust_daily.rename(columns={'측정일시': '날짜'}, inplace=True)

# 과거부터 최신순으로 날짜 정렬
df_dust_daily = df_dust_daily.sort_values(by='날짜').reset_index(drop=True)


# =========================================================
# [4단계] 최종 통합 파일 저장
# =========================================================
df_dust_daily.to_csv('서울시_일별_미세먼지_통합본(19-25).csv', index=False, encoding='utf-8-sig')

print("\ 연도별 컬럼명 불일치 버그 해결 및 전처리 최종 완료!")
print(" '서울시_일별_미세먼지_통합본(19-25).csv' 파일이 정상 생성되었습니다.")
display(df_dust_daily.head(10))

 -> 총 7개의 파일 컬럼명 자동 통일 및 합체 완료!
\ 연도별 컬럼명 불일치 버그 해결 및 전처리 최종 완료!
 '서울시_일별_미세먼지_통합본(19-25).csv' 파일이 정상 생성되었습니다.


<>:58: SyntaxWarning: invalid escape sequence '\ '
<>:58: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_450/51807611.py:58: SyntaxWarning: invalid escape sequence '\ '
  print("\ 연도별 컬럼명 불일치 버그 해결 및 전처리 최종 완료!")


,날짜,미세먼지,초미세먼지
0,2019-01-01,37.6,24.8
1,2019-01-02,34.4,22.2
2,2019-01-03,39.2,23.9
3,2019-01-04,59.9,41.6
4,2019-01-05,64.2,40.9
5,2019-01-06,49.0,23.4
6,2019-01-07,56.8,37.1
7,2019-01-08,40.6,18.3
8,2019-01-09,51.4,22.4
9,2019-01-10,57.6,33.9


In [16]:
import pandas as pd



# 1. 기상 데이터 불러오기 (인코딩은 기상청 기본형인 cp949)
df_weather = pd.read_csv('OBS_ASOS_DD_20260518163944.csv', encoding='cp949')

# 2. 컬럼명 표준화 기획
rename_dict = {
    '일시': '날짜',
    '최저기온(°C)': '최저기온',
    '최고기온(°C)': '최고기온',
    '일강수량(mm)': '강수량'
}

df_weather.rename(columns={k: v for k, v in rename_dict.items() if k in df_weather.columns}, inplace=True)

# 필요한 핵심 컬럼만 슬라이싱
df_weather = df_weather[['날짜', '최저기온', '최고기온', '강수량']]


# 3. 기상청 특유의 비 안 온 날 결측치 ➔ 0.0mm로 완벽 보간
df_weather['강수량'] = df_weather['강수량'].fillna(0.0)


# 4. 날짜 포맷 표준화 (2019-01-01 모양으로 세탁) 및 정렬
df_weather['날짜'] = pd.to_datetime(df_weather['날짜']).dt.strftime('%Y-%m-%d')
df_weather = df_weather.sort_values(by='날짜').reset_index(drop=True)


# 5. 최종 전처리본 저장
df_weather.to_csv('서울시_일별_날씨_통합본(19-25).csv', index=False, encoding='utf-8-sig')

print("\n 데이터 다이어트 및 전처리 완료!")
print(" '서울시_일별_날씨_통합본(19-25).csv' 파일이 생성되었습니다.")
display(df_weather.head(10))


 데이터 다이어트 및 전처리 완료!
 '서울시_일별_날씨_통합본(19-25).csv' 파일이 생성되었습니다.


,날짜,최저기온,최고기온,강수량
0,2019-01-01,-8.2,-0.6,0.0
1,2019-01-02,-8.8,0.2,0.0
2,2019-01-03,-8.4,3.2,0.0
3,2019-01-04,-6.2,4.1,0.0
4,2019-01-05,-5.5,1.1,0.0
5,2019-01-06,-6.3,2.7,0.0
6,2019-01-07,-6.2,3.1,0.0
7,2019-01-08,-7.2,0.5,0.0
8,2019-01-09,-9.4,1.3,0.0
9,2019-01-10,-4.5,3.0,0.0


# **최종데이터셋**

In [19]:
import pandas as pd

# 1. 전처리가 끝난 3개의 파일 불러오기
# (팀원들이 저장한 실제 파일명으로 각각 매칭해주세요)
df_y = pd.read_csv('최종_완벽본_일별무임승차량(19-25).csv')
df_weather = pd.read_csv('서울시_일별_날씨_통합본(19-25).csv')
df_dust = pd.read_csv('서울시_일별_미세먼지_통합본(19-25).csv')

# 혹시 모를 날짜 모양 불일치를 방지하기 위해 형식을 다시 한 번 통일 (안전장치)
df_y['날짜'] = pd.to_datetime(df_y['날짜']).dt.strftime('%Y-%m-%d')
df_weather['날짜'] = pd.to_datetime(df_weather['날짜']).dt.strftime('%Y-%m-%d')
df_dust['날짜'] = pd.to_datetime(df_dust['날짜']).dt.strftime('%Y-%m-%d')

# 2. 1차 병합: 정답지(Y) 데이터에 날씨(X1) 데이터 붙이기
df_merged = pd.merge(df_y, df_weather, on='날짜', how='left')

# 3. 2차 병합: 위 결과에 미세먼지(X2) 데이터까지 최종적으로 붙이기
df_final = pd.merge(df_merged, df_dust, on='날짜', how='left')

# 4. 날짜순으로 예쁘게 오름차순 정렬
df_final = df_final.sort_values(by='날짜').reset_index(drop=True)

# 5. 모델 학습용 최종 데이터셋 저장
df_final.to_csv('XGBoost_학습용_최종데이터셋(19-25).csv', index=False, encoding='utf-8-sig')

print("\ 머신러닝 직전 최종 데이터셋이 완성되었습니다.")
display(df_final.head(10))

\ 머신러닝 직전 최종 데이터셋이 완성되었습니다.


<>:26: SyntaxWarning: invalid escape sequence '\ '
<>:26: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_450/978872170.py:26: SyntaxWarning: invalid escape sequence '\ '
  print("\ 머신러닝 직전 최종 데이터셋이 완성되었습니다.")


,날짜,일별무임승차량,최저기온,최고기온,강수량,미세먼지,초미세먼지
0,2019-01-01,324283,-8.2,-0.6,0.0,37.6,24.8
1,2019-01-02,722134,-8.8,0.2,0.0,34.4,22.2
2,2019-01-03,765807,-8.4,3.2,0.0,39.2,23.9
3,2019-01-04,784641,-6.2,4.1,0.0,59.9,41.6
4,2019-01-05,633946,-5.5,1.1,0.0,64.2,40.9
5,2019-01-06,475875,-6.3,2.7,0.0,49.0,23.4
6,2019-01-07,791914,-6.2,3.1,0.0,56.8,37.1
7,2019-01-08,795932,-7.2,0.5,0.0,40.6,18.3
8,2019-01-09,755806,-9.4,1.3,0.0,51.4,22.4
9,2019-01-10,798343,-4.5,3.0,0.0,57.6,33.9
